# makable Shorts: the intro and the recording, joined

The whole video from two files: the intro (Omni's clip of you, in your voice), then a screen recording (narrated, or silent), each with its own sound, in one 1080×1920 MP4. A clip the frame's shape fills it; any other (a desktop recording in a vertical video) is shown whole over a blurred, darker copy of itself. The sound is evened out to -14 LUFS. No captions or music: the Shorts notebook adds those.

It runs `merge_intro.py` and the video engine it uses (`shorts.compose`), the same files as on the command line. They need only ffmpeg, so the CPU runtime is enough.

**Before you start**, have the two videos on your computer (or on Drive):

1. The intro avatar video, in your voice: section 4 of `voice_from_script.ipynb` makes it, as `intro/voiced.mp4` in the run's folder. Or Omni's own clip, with its sound in your voice as `INTRO_AUDIO`.
2. The screen recording, narrated or silent. `narrate_recording.ipynb` narrates a recording with scenes spoken already.

Section 1 asks you to upload the intro, section 2 the screen recording, and section 3 joins them. An upload stays in this session only; the video goes on Drive, in `RUN`.

## Setup

In [ ]:
import glob
import os
import sys
from pathlib import Path

from google.colab import drive, files
from IPython.display import Video, display

drive.mount("/content/drive")

HOME = Path("/content/drive/MyDrive/makable-shorts")
RUN = HOME / "runs" / "crumb-and-co"  # the video's folder: the intro in your voice is its intro/voiced.mp4


In [ ]:
os.makedirs("/content/shorts", exist_ok=True)


In [ ]:
%%writefile /content/shorts/__init__.py
"""makable Shorts: a 1-minute vertical video about what changed in makable, made by stages that share a folder.

Each stage is a program of its own with its own packages (`shorts.stages`), reading and writing a video's run
folder under the names in `shorts.run`, the contract between them: record (the footage) -> write-script
(Gemini) -> film-intro (Gemini Omni) -> revoice-intro and speak (your voice, on a GPU) -> assemble (ffmpeg).
`shorts.orchestrator` runs what's due, each stage in a process of its own. The other modules are what the
stages use, and know nothing of run folders.
"""


In [ ]:
%%writefile /content/shorts/media.py
"""Small wrappers around ffmpeg and ffprobe."""

import json
import re
import subprocess
from pathlib import Path


def ffmpeg(*args: str | Path) -> None:
    """Runs ffmpeg (overwriting outputs) and raises with its error text if it fails."""
    command = ["ffmpeg", "-hide_banner", "-loglevel", "error", "-y", *map(str, args)]
    result = subprocess.run(command, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"ffmpeg failed: {result.stderr.strip()}")


def duration(path: Path) -> float:
    """Returns a media file's length in seconds."""
    result = subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "json", str(path)],
        capture_output=True,
        text=True,
        check=True,
    )
    return float(json.loads(result.stdout)["format"]["duration"])


def frame_size(path: Path) -> tuple[int, int]:
    """Returns a video's width and height."""
    result = subprocess.run(
        [
            "ffprobe",
            "-v",
            "error",
            "-select_streams",
            "v:0",
            "-show_entries",
            "stream=width,height",
            "-of",
            "json",
            str(path),
        ],
        capture_output=True,
        text=True,
        check=True,
    )
    stream = json.loads(result.stdout)["streams"][0]
    return stream["width"], stream["height"]


def has_sound(path: Path) -> bool:
    """Whether a media file has an audio stream."""
    result = subprocess.run(
        ["ffprobe", "-v", "error", "-select_streams", "a", "-show_entries", "stream=index", "-of", "json", str(path)],
        capture_output=True,
        text=True,
        check=True,
    )
    return bool(json.loads(result.stdout).get("streams"))


def changes(path: Path, fps: int, pixel: int = 8, area: float = 0.15) -> list[float]:
    """Returns when a video's picture changes (seconds into it): each frame in which more than `area` % of
    the pixels got more than `pixel` (of 255) lighter or darker since the frame before."""
    # The changed pixels, averaged over a 45x80 grid (16x16-pixel cells of a 720x1280 frame): ffmpeg does
    # it 8x faster than signalstats. accurate_rnd, or a cell with nothing changed comes out 1, not 0.
    cells = 45 * 80
    result = subprocess.run(
        [
            "ffmpeg",
            "-hide_banner",
            "-loglevel",
            "error",
            "-i",
            str(path),
            "-an",
            "-vf",
            f"format=gray,tblend=all_mode=difference,lutyuv=y='if(gt(val,{pixel}),255,0)',"
            "scale=45:80:flags=area+accurate_rnd",
            "-f",
            "rawvideo",
            "-pix_fmt",
            "gray",
            "-",
        ],
        capture_output=True,
    )
    if result.returncode != 0:
        raise RuntimeError(f"ffmpeg failed: {result.stderr.decode().strip()}")
    grids = result.stdout
    return [
        (i // cells + 1) / fps  # tblend's frame n is how frame n + 1 differs from frame n
        for i in range(0, len(grids), cells)
        if sum(grids[i : i + cells]) / (255 * cells) * 100 > area
    ]


def speech_span(path: Path, noise_db: int = -35, min_silence: float = 0.25) -> tuple[float, float]:
    """Returns when the speech in an audio file starts and ends, skipping silence at either end."""
    result = subprocess.run(
        [
            "ffmpeg",
            "-hide_banner",
            "-i",
            str(path),
            "-af",
            f"silencedetect=noise={noise_db}dB:d={min_silence}",
            "-f",
            "null",
            "-",
        ],
        capture_output=True,
        text=True,
    )
    total = duration(path)
    starts = [float(s) for s in re.findall(r"silence_start: (-?[\d.]+)", result.stderr)]
    ends = [float(s) for s in re.findall(r"silence_end: (-?[\d.]+)", result.stderr)]
    silences = list(zip(starts, ends + [total] * (len(starts) - len(ends)), strict=False))
    begin = silences[0][1] if silences and silences[0][0] <= 0.05 else 0.0
    end = silences[-1][0] if silences and silences[-1][1] >= total - 0.05 else total
    return (begin, end) if end > begin else (0.0, total)


In [ ]:
%%writefile /content/shorts/script.py
"""The script: an LLM turns what changed, or what a recording shows, into what's said and shown."""

from typing import Any

from pydantic import BaseModel, Field
from pydantic.json_schema import SkipJsonSchema

TEXT_MODEL = "gemini-3.8-flash"  # free tier in the Gemini API; on Agent Platform, a fraction of a cent a script
WORDS_PER_SECOND = 2.5  # how fast the narration is spoken, near enough
NARRATION = 50  # seconds of narrated recording after the 10-second intro
# The languages a script can be in, by the code Chatterbox speaks them with, and how they're written.
LANGUAGES = {
    "en": "English",
    "hi": (
        "everyday spoken Hindi, in Devanagari, the way a friendly Indian tech YouTuber talks. The English words "
        "Indians say anyway (website, preview, button, menu) stay English, written in Devanagari (वेबसाइट, प्रीव्यू). "
        "Write {product_name} in Latin letters"
    ),
}
# How the narrator sounds, by name. "hype" is the excited style of Shorts like FactoHolic's: a creator showing a
# friend something they just found, fast, with exclamations and a turn before the best part.
TONES = {
    "friendly": "Write like a person talking: short sentences, plain words",
    "hype": (
        "Write like a creator excitedly showing a friend something they just found: short, punchy sentences, plain "
        "words, an exclamation mark where something good happens, words that pull the viewer in (look, just, wait) "
        'and, before the best part, a turn like "wait, that\'s not even the best part"'
    ),
}

SYSTEM = """You write scripts for 1-minute vertical videos (YouTube Shorts, Instagram Reels) about {product}.

The video has two parts:
1. A 10-second intro: the presenter talks to the camera. `intro_line` is what they say: one hook of 14 to 20 words that makes people want to see the change. `intro_setting` describes the shot in one sentence (place, light, framing), not the person.
2. A narrated screen recording: 4 to 6 `scenes`, 110 to 130 words in all. Each scene is one or two short sentences, said over one recording named in `footage`. Use only the recording names you're given.

{tone}. No hashtags or emojis in anything that's said. End by asking people to try {product_name}.
Also write a YouTube `title` (under 70 characters) and a short `description` (2 or 3 sentences).
Write everything that's said, the `title` and the `description` in {language}."""

STEPS_SYSTEM = """You write scripts for 1-minute vertical videos (YouTube Shorts, Instagram Reels) about {product}.

The video has two parts:
1. A 10-second intro: the presenter talks to the camera. `intro_line` is what they say: one hook of 14 to 20 words that makes people want to see what happens. `intro_setting` describes the shot in one sentence (place, light, framing), not the person.
2. A narrated screen recording: 3 to 8 `scenes`, about {words} words in all. You're given what happens in each recording, step by step, with when (seconds into it). Each scene is said over one stretch of one recording: `footage` names the recording, `start` and `end` (seconds) are where the stretch begins and ends. The narration is spoken at about {rate:g} words a second, so give each scene about {rate:g} words for each second of its stretch (6 seconds: about {six} words), and pick stretches that add up to about {seconds} seconds.

Tell the story in order. The stretches play one after another and never overlap, so leave out what doesn't move the story along: retries, error messages, a name that was already taken, moments where nothing new happens. Say what's on screen while it's on screen.

A program used the site to make the recording, and the steps are in its words ("Click element [471]", "Wait 30 seconds"). Never mention it, its waits or its retries: narrate as if you were showing the viewer. A step that only waits shows what the page did meanwhile (progress, a preview loading), cut down to the moments it changed. Typed text appears on screen as it's typed. You can use facts from the program's final report, such as how long the build took or what the finished page has, but claim nothing the steps and the report don't back up: not "instantly" for something that took 30 seconds.

{tone}. No hashtags or emojis in anything that's said. End by asking people to try {product_name}.
Also write a YouTube `title` (under 70 characters) and a short `description` (2 or 3 sentences).
Write everything that's said, the `title` and the `description` in {language}."""


class Scene(BaseModel):
    text: str = Field(description="What the narrator says over this recording: one or two short sentences.")
    footage: str = Field(description="The name of the recording shown while it's said.")
    # Where in the recording the scene is, in seconds, for a script written from steps. Without them, the
    # recording plays on from where the scene before on it stopped. Not asked for by `write_script`.
    start: SkipJsonSchema[float | None] = None
    end: SkipJsonSchema[float | None] = None


class Script(BaseModel):
    intro_line: str = Field(description="What the presenter says to the camera, 14 to 20 words.")
    intro_setting: str = Field(description="The intro shot in one sentence: place, light, framing.")
    scenes: list[Scene]
    title: str
    description: str
    language: SkipJsonSchema[str] = "en"  # what it's written in (a code in LANGUAGES), so it's spoken in it


class _TimedScene(Scene):
    text: str = Field(description="What the narrator says over this stretch of the recording.")
    start: float = Field(description="Where the stretch begins, in seconds into the recording.")
    end: float = Field(description="Where the stretch ends, in seconds into the recording.")


class _TimedScript(Script):
    scenes: list[_TimedScene]


def _ask(client: Any, model: str, system: str, contents: str, schema: type[BaseModel]) -> Script:
    """Asks the model for a script, as JSON matching `schema`."""
    from google.genai import types

    response = client.models.generate_content(
        model=model,
        contents=contents,
        config=types.GenerateContentConfig(
            system_instruction=system,
            response_mime_type="application/json",
            response_schema=schema,
        ),
    )
    return Script.model_validate_json(response.text)


def written_in(language: str, product_name: str) -> str:
    """How the prompt asks for `language` (a code in `LANGUAGES`)."""
    if language not in LANGUAGES:
        raise ValueError(f"No script language {language!r}: one of {', '.join(LANGUAGES)}")
    return LANGUAGES[language].format(product_name=product_name)


def in_tone(tone: str) -> str:
    """How the prompt asks for `tone` (a name in `TONES`)."""
    if tone not in TONES:
        raise ValueError(f"No tone {tone!r}: one of {', '.join(TONES)}")
    return TONES[tone]


def write_script(
    client: Any,
    changes: str,
    footage: list[str],
    *,
    language: str = "en",
    tone: str = "friendly",
    product: str = "makable, an AI website builder",
    model: str = TEXT_MODEL,
) -> Script:
    """Asks the model for the script, as JSON matching `Script`.

    Args:
        client: A `google.genai.Client`.
        changes: What changed: commit messages, release notes, or a few lines about the feature.
        footage: The names of the recordings the scenes can use.
        language: What it's written (and later spoken) in: a code in `LANGUAGES`, "en" or "hi".
        tone: How it sounds: a name in `TONES`, "friendly" or "hype" (excited, the way Shorts creators talk).
        product: The product, as the narrator would introduce it.
        model: The Gemini text model.
    """
    product_name = product.split(",")[0]
    script = _ask(
        client,
        model,
        SYSTEM.format(
            product=product,
            product_name=product_name,
            language=written_in(language, product_name),
            tone=in_tone(tone),
        ),
        f"What changed:\n{changes}\n\nRecordings you can use: {', '.join(footage)}",
        Script,
    )
    script.language = language
    for index, scene in enumerate(script.scenes):
        if scene.footage not in footage:
            scene.footage = footage[index % len(footage)]
    return script


def recording_length(steps: list[dict[str, Any]]) -> float:
    """How long a recording is, in seconds: until its last step ends."""
    return max((step["end"] for step in steps), default=0.0)


def timeline(name: str, steps: list[dict[str, Any]]) -> str:
    """A recording's steps for the prompt, one line each: when, what the step set out to do, and what it did."""
    lines = [f'Recording "{name}" ({recording_length(steps):.1f} s):']
    for step in steps:
        when = f"{step['start']:.1f}-{step['end']:.1f} s"
        kinds = {kind for action in step["actions"] for kind in action}
        if "done" in kinds:  # its account is the run's final report
            lines.append(f"{when}: the end. The program's final report:\n" + "\n".join(step["did"]))
            continue
        did = "; ".join(" ".join(item.split()) for item in step["did"] if item)
        waits = " (only the moments the page changed are kept)" if kinds == {"wait"} else ""
        lines.append(f"{when}: {step['goal']} Did: {did}{waits}")
    return "\n".join(lines)


def write_script_from_steps(
    client: Any,
    recordings: dict[str, list[dict[str, Any]]],
    *,
    changes: str = "",
    language: str = "en",
    tone: str = "friendly",
    product: str = "makable, an AI website builder",
    model: str = TEXT_MODEL,
) -> Script:
    """Asks the model for a script narrating recordings made by `browse.record_task`, from their steps.

    Each scene is said over a stretch of a recording (`start` to `end`), picked to tell the story and
    sized to the narration, so what's said matches what's on screen. Retries and errors are left out.

    Args:
        client: A `google.genai.Client`.
        recordings: Each recording's name and its steps, as saved in its `.steps.json` (`browse.load_steps`,
            or the JSON itself). Use the steps of the video that's shown: `cut_pauses`' for `-tight.mp4`.
        changes: What changed, if the video is about a change; the steps are enough otherwise.
        language: What it's written (and later spoken) in: a code in `LANGUAGES`, "en" or "hi".
        tone: How it sounds: a name in `TONES`, "friendly" or "hype" (excited, the way Shorts creators talk).
        product: The product, as the narrator would introduce it.
        model: The Gemini text model.
    """
    recordings = {
        name: [step if isinstance(step, dict) else vars(step) for step in steps] for name, steps in recordings.items()
    }
    lengths = {name: recording_length(steps) for name, steps in recordings.items()}
    seconds = round(min(NARRATION, sum(lengths.values())))
    product_name = product.split(",")[0]
    system = STEPS_SYSTEM.format(
        product=product,
        product_name=product_name,
        language=written_in(language, product_name),
        tone=in_tone(tone),
        words=round(seconds * WORDS_PER_SECOND),
        rate=WORDS_PER_SECOND,
        six=round(6 * WORDS_PER_SECOND),
        seconds=seconds,
    )
    contents = "\n\n".join(timeline(name, steps) for name, steps in recordings.items())
    if changes:
        contents = f"What changed:\n{changes}\n\n{contents}"
    script = _ask(client, model, system, contents, _TimedScript)
    script.language = language
    names = list(recordings)
    for index, scene in enumerate(script.scenes):
        if scene.footage not in recordings:
            scene.footage = names[index % len(names)]
        length = lengths[scene.footage]
        scene.start = min(max(scene.start or 0.0, 0.0), length)
        scene.end = min(max(scene.end or length, scene.start), length)
    return script


In [ ]:
%%writefile /content/shorts/captions.py
"""Captions: a few words at a time, spread across each spoken part, drawn with Pillow.

Drawn as images rather than with ffmpeg's subtitles filter, because not every ffmpeg build has
libass (Homebrew's doesn't). The pictures become one video stream through an ffconcat list.
"""

import re
import urllib.request
from dataclasses import dataclass
from pathlib import Path

from PIL import Image, ImageDraw, ImageFont

SIZE = (1080, 1920)  # the frame the captions are drawn for, by default
FONT_URL = "https://github.com/google/fonts/raw/main/ofl/montserrat/Montserrat%5Bwght%5D.ttf"
CENTER_Y = 0.62  # above the title and buttons Shorts and Reels draw over the bottom


@dataclass
class Caption:
    text: str
    start: float
    end: float


def chunk_words(text: str, max_words: int = 3, max_chars: int = 16) -> list[str]:
    """Splits text into short chunks, ending one at every punctuation mark."""
    chunks: list[str] = []
    current: list[str] = []
    for word in text.split():
        if current and (len(current) == max_words or len(" ".join([*current, word])) > max_chars):
            chunks.append(" ".join(current))
            current = []
        current.append(word)
        if re.search(r"[.!?,;:]$", word):
            chunks.append(" ".join(current))
            current = []
    if current:
        chunks.append(" ".join(current))
    return chunks


def timed(text: str, start: float, end: float) -> list[Caption]:
    """Spreads the chunks of `text` over [start, end], each for a time in proportion to its length."""
    chunks = chunk_words(text)
    weights = [len(chunk) + 3 for chunk in chunks]
    captions = []
    at = start
    for chunk, weight in zip(chunks, weights, strict=True):
        length = (end - start) * weight / sum(weights)
        captions.append(Caption(chunk, at, at + length))
        at += length
    return captions


def load_font(cache: Path, size: int = 88) -> ImageFont.FreeTypeFont:
    """Montserrat ExtraBold, downloaded once into `cache`."""
    path = cache / "Montserrat.ttf"
    if not path.exists():
        cache.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(FONT_URL, path)
    font = ImageFont.truetype(str(path), size)
    font.set_variation_by_axes([800])
    return font


def draw(text: str, font: ImageFont.FreeTypeFont, size: tuple[int, int] = SIZE) -> Image.Image:
    """One caption: white with a black outline, wrapped to the frame, on a transparent frame."""
    width, height = size
    image = Image.new("RGBA", size)
    pen = ImageDraw.Draw(image)
    lines: list[str] = []
    for word in text.split():
        if lines and pen.textlength(f"{lines[-1]} {word}", font=font) <= width - 160:
            lines[-1] = f"{lines[-1]} {word}"
        else:
            lines.append(word)
    pen.multiline_text(
        (width / 2, height * CENTER_Y),
        "\n".join(lines),
        font=font,
        anchor="mm",
        align="center",
        spacing=14,
        fill="white",
        stroke_width=9,
        stroke_fill="black",
    )
    return image


def render(captions: list[Caption], workdir: Path, font: ImageFont.FreeTypeFont, size: tuple[int, int] = SIZE) -> Path:
    """Draws each caption and returns an ffconcat list that shows each one for its time.

    Read it with `-f concat -i <list>`; gaps between captions show a blank frame.
    """
    workdir.mkdir(parents=True, exist_ok=True)
    blank = workdir / "blank.png"
    Image.new("RGBA", size).save(blank)
    entries = ["ffconcat version 1.0"]
    at = 0.0
    for index, caption in enumerate(captions):
        if caption.start > at + 0.001:
            entries += [f"file {blank.name}", f"duration {caption.start - at:.3f}"]
        image = workdir / f"{index:03d}.png"
        draw(caption.text, font, size).save(image)
        entries += [f"file {image.name}", f"duration {caption.end - caption.start:.3f}"]
        at = caption.end
    entries.append(f"file {blank.name}")  # the last entry's duration is ignored
    listing = workdir / "captions.ffconcat"
    listing.write_text("\n".join(entries) + "\n")
    return listing


In [ ]:
%%writefile /content/shorts/compose.py
"""Putting the video together with ffmpeg: parts one after another, with captions and music.

A part is footage on screen while something is heard: the intro and its speech, a scene's stretch of a
recording and its narration, or a recording and its own sound. A stretch with an end is sped up or slowed
down to last as long as its part, so what's said is on screen while it's said; without one, the footage
plays at its own speed from `start`, looping if it's shorter. Footage the frame's shape fills it, cropped a
little; any other is shown whole over a blurred, darker copy of itself.
"""

from dataclasses import dataclass
from pathlib import Path

from .media import ffmpeg, frame_size, has_sound

FPS = 30
SIZE = (1080, 1920)  # YouTube Shorts and Instagram Reels
GAP = 0.35  # seconds of pause after each scene's narration
TAIL = 1.0  # extra seconds on the last scene
SAME_SHAPE = 0.05  # how far footage's width:height can be from the frame's (a share of it) and still fill it
FONT_CACHE = Path.home() / ".cache" / "makable-shorts"


@dataclass
class Part:
    """A stretch of the video: `footage` on screen for `length` seconds while `speech` plays from its start."""

    footage: Path
    length: float  # seconds on screen
    speech: Path | None = None  # what's heard (the footage itself, for its own sound); None: silence
    text: str = ""  # what's said, for the captions
    said: tuple[float, float] | None = None  # when in `speech` the words start and end; None: all along
    start: float = 0.0  # where in the footage to start
    end: float | None = None  # where to end, fitting start to end to `length`; None: at the footage's own speed

    @property
    def speed(self) -> float:
        return 1.0 if self.end is None else (self.end - self.start) / self.length


def whole_frames(seconds: float) -> float:
    """Rounds to whole frames, so the parts' lengths add up exactly."""
    return round(seconds * FPS) / FPS


def even(size: tuple[int, int]) -> tuple[int, int]:
    """The nearest frame size at or under `size` that yuv420p can have: even numbers."""
    return size[0] // 2 * 2, size[1] // 2 * 2


def fills(size: tuple[int, int], frame: tuple[int, int]) -> bool:
    """Whether footage of `size` is near enough the frame's shape to fill it, cropped a little."""
    return abs(size[0] / size[1] - frame[0] / frame[1]) <= SAME_SHAPE * frame[0] / frame[1]


def fit(size: tuple[int, int], frame: tuple[int, int]) -> str:
    """The filters that scale footage of `size` to the frame."""
    width, height = frame
    if fills(size, frame):
        return f"scale={width}:{height}:force_original_aspect_ratio=increase,crop={width}:{height}"
    small = f"{width // 4}:{height // 4}"
    blur = max(1, min(12, min(width, height) // 16 - 1))  # boxblur's radius must stay under a quarter of the
    # small copy's size (its colour is at half size): 12 for a 1080x1920 frame
    return (
        f"split[a][b];"
        f"[a]scale={small}:force_original_aspect_ratio=increase,crop={small},boxblur={blur}:2,scale={width}:{height},"
        f"eq=brightness=-0.12[bg];"
        f"[b]scale={width}:{height}:force_original_aspect_ratio=decrease[fg];"
        f"[bg][fg]overlay=(W-w)/2:(H-h)/2"
    )


def render_part(part: Part, out: Path, frame: tuple[int, int] = SIZE) -> Path:
    """Encodes one part's picture (no sound), holding the last frame if the footage runs out."""
    if part.end is None:
        source = ["-stream_loop", "-1", "-ss", f"{part.start:.3f}", "-i", part.footage]
        timing = ""
    else:
        source = ["-ss", f"{part.start:.3f}", "-to", f"{part.end:.3f}", "-i", part.footage]
        timing = f"setpts=(PTS-STARTPTS)/{part.speed:.6f},"
    ffmpeg(
        *source,
        "-filter_complex",
        f"[0:v]{timing}fps={FPS},{fit(frame_size(part.footage), frame)},setsar=1,format=yuv420p,"
        f"tpad=stop_mode=clone:stop_duration={part.length:.3f}[v]",
        "-map", "[v]", "-t", f"{part.length:.3f}", "-an",
        "-c:v", "libx264", "-preset", "veryfast", "-crf", "18", out,
    )  # fmt: skip
    return out


def soundtrack(parts: list[Part], out: Path, music: Path | None = None, music_volume: float = 0.12) -> Path:
    """Lays each part's speech at its start (padded to its length), adds music, and evens out the loudness."""
    inputs: list[str | Path] = []
    chains = []
    for index, part in enumerate(parts):
        if part.speech is not None and has_sound(part.speech):
            inputs += ["-i", part.speech]
        else:
            inputs += ["-f", "lavfi", "-i", "anullsrc=channel_layout=stereo:sample_rate=48000"]
        chains.append(
            f"[{index}:a]aresample=48000,aformat=channel_layouts=stereo,apad,atrim=0:{part.length:.3f}[p{index}]"
        )
    labels = "".join(f"[p{index}]" for index in range(len(parts)))
    graph = ";".join(chains) + f";{labels}concat=n={len(parts)}:v=0:a=1[voice]"
    mixed = "[voice]"
    if music is not None:
        total = sum(part.length for part in parts)
        inputs += ["-stream_loop", "-1", "-i", music]
        graph += (
            f";[{len(parts)}:a]aresample=48000,aformat=channel_layouts=stereo,atrim=0:{total:.3f},"
            f"volume={music_volume},afade=t=out:st={max(total - 2, 0):.3f}:d=2[music]"
            ";[voice][music]amix=inputs=2:duration=first:normalize=0[mixed]"
        )
        mixed = "[mixed]"
    # -14 LUFS is what YouTube and Instagram play at. loudnorm works at 192 kHz, hence the resample.
    graph += f";{mixed}loudnorm=I=-14:TP=-1.5:LRA=11,aresample=48000[out]"
    ffmpeg(*inputs, "-filter_complex", graph, "-map", "[out]", "-c:a", "pcm_s16le", out)
    return out


def compose(
    parts: list[Part],
    workdir: Path,
    out: Path,
    *,
    size: tuple[int, int] = SIZE,
    music: Path | None = None,
    captions: bool = True,
) -> Path:
    """Makes the MP4 from the parts, in order: `size` (even numbers), captions of what's said, and music."""
    workdir.mkdir(parents=True, exist_ok=True)
    pictures = [render_part(part, workdir / f"part-{index}.mp4", size) for index, part in enumerate(parts)]
    listing = workdir / "parts.ffconcat"
    listing.write_text("ffconcat version 1.0\n" + "".join(f"file {picture.name}\n" for picture in pictures))
    audio = soundtrack(parts, workdir / "soundtrack.wav", music)

    inputs: list[str | Path] = ["-f", "concat", "-i", listing, "-i", audio]
    picture = ["-map", "0:v"]
    if captions:
        from .captions import Caption, load_font, render, timed  # Pillow, only for captions

        timings: list[Caption] = []
        at = 0.0
        for part in parts:
            if part.text:
                begin, end = part.said or (0.0, part.length)
                timings += timed(part.text, at + begin, at + min(end, part.length))
            at += part.length
        inputs += ["-f", "concat", "-i", render(timings, workdir / "captions", load_font(FONT_CACHE), size)]
        picture = ["-filter_complex", "[0:v][2:v]overlay=0:0:eof_action=pass,format=yuv420p[v]", "-map", "[v]"]
    ffmpeg(
        *inputs, *picture, "-map", "1:a",
        "-c:v", "libx264", "-preset", "medium", "-crf", "20", "-pix_fmt", "yuv420p", "-r", str(FPS),
        "-c:a", "aac", "-b:a", "192k", "-movflags", "+faststart", "-t", f"{sum(part.length for part in parts):.3f}", out,
    )  # fmt: skip
    return out


In [ ]:
%%writefile /content/merge_intro.py
"""Puts the intro (Omni's clip of you) before a screen recording: the whole video, start to end.

The intro plays first, then the recording, each with its own sound: the intro's speech, then the recording's
(say, the narration narrate_recording.py put over it). A recording with no sound stays silent. Omni's clip has
Omni's voice: pass the intro in your voice (a run's intro/voiced.mp4), or Omni's clip with `--intro-audio` (the
intro's sound in your voice). The picture is 1080x1920, like the Shorts (`--size 1920x1080` for a wide video):
a clip the frame's shape fills it, any other is shown whole over a blurred, darker copy of itself. The sound is
evened out to -14 LUFS, and there are no captions or music. For the whole video with those, bring the files
into a run folder instead (`python -m shorts import`, see the README).

    python3 merge_intro.py runs/crumb-and-co/intro/voiced.mp4 makable-tight-narrated.mp4 \\
        [--intro-audio intro-in-your-voice.wav] [--size 1080x1920] [-o final.mp4]

Needs ffmpeg and the shorts package next to this file (in Colab: merge_intro.ipynb writes both).
"""

import argparse
import tempfile
from pathlib import Path

from shorts.compose import SIZE, Part, compose, fills, whole_frames
from shorts.media import duration, frame_size, has_sound


def merge(
    intro: Path, recording: Path, out: Path, *, intro_audio: Path | None = None, size: tuple[int, int] = SIZE
) -> Path:
    """Makes `out`: the intro, then the recording, in one frame size, each with its own sound."""
    if out.resolve() in {intro.resolve(), recording.resolve()}:
        raise SystemExit(f"{out} is one of the videos going in: pick another name with -o")
    parts = []
    for name, clip, sound in [("Intro", intro, intro_audio or intro), ("Recording", recording, recording)]:
        shape, length = frame_size(clip), whole_frames(duration(clip))
        fitted = "fills the frame" if fills(shape, size) else "whole, over a blurred copy"
        heard = f"sound from {sound.name}" if has_sound(sound) else "silent"
        print(f"{name}: {clip.name}, {shape[0]}x{shape[1]}, {length:.2f} s, {fitted}, {heard}")
        parts.append(Part(clip, length, sound))
    with tempfile.TemporaryDirectory() as work:
        compose(parts, Path(work), out, size=size, captions=False)
    return out


def frame_size_arg(text: str) -> tuple[int, int]:
    """--size's value, such as 1080x1920: even numbers, which yuv420p needs."""
    try:
        width, height = map(int, text.lower().split("x"))
    except ValueError:
        raise argparse.ArgumentTypeError(f"{text!r} isn't a size like 1080x1920") from None
    if width <= 0 or height <= 0 or width % 2 or height % 2:
        raise argparse.ArgumentTypeError(f"{text}: the width and height must be even")
    return width, height


def main() -> None:
    parser = argparse.ArgumentParser(description="Put the intro before a screen recording, for the whole video.")
    parser.add_argument("intro", type=Path, help="the intro clip Omni filmed (in your voice, or with --intro-audio)")
    parser.add_argument("recording", type=Path, help="the screen recording, narrated or silent")
    parser.add_argument("--intro-audio", type=Path, help="the intro's sound in your voice, in place of the clip's")
    parser.add_argument(
        "--size", type=frame_size_arg, default=SIZE, help="the video's width x height (default: 1080x1920)"
    )
    parser.add_argument("-o", "--out", type=Path, help="the video to make (default: <recording>-with-intro.mp4)")
    args = parser.parse_args()
    out = args.out or args.recording.with_name(f"{args.recording.stem}-with-intro.mp4")
    merge(args.intro, args.recording, out, intro_audio=args.intro_audio, size=args.size)
    print(f"Made {out} ({duration(out):.1f} s)")


if __name__ == "__main__":
    main()


In [ ]:
%load_ext autoreload
%autoreload 2
sys.path.insert(0, "/content")

from merge_intro import duration, frame_size, merge


def chosen(setting, what):
    """A file: `setting` itself, the newest file matching it (a pattern), or one you upload (None)."""
    if setting is None:
        print(f"Choose {what} to upload:")
        uploads = files.upload()  # saved in /content
        if not uploads:
            raise FileNotFoundError(f"No {what}: upload one, or set it to a file on Drive")
        return Path(next(iter(uploads))).resolve()
    if Path(setting).exists():
        return Path(setting)
    matches = sorted(glob.glob(str(setting)), key=os.path.getmtime)
    if not matches:
        raise FileNotFoundError(f"Nothing on Drive matches {setting}")
    return Path(matches[-1])


## 1. The intro avatar video

Run the cell, then choose the intro on your computer. Or set `INTRO` to a file on Drive, such as `RUN / "intro" / "voiced.mp4"`, or a pattern (the newest match), and nothing is uploaded.

In [ ]:
INTRO = None  # None: upload it; or a file on Drive (such as RUN / "intro" / "voiced.mp4"), or a pattern
INTRO_AUDIO = None  # with Omni's own clip as INTRO: its sound in your voice (a WAV), in place of the clip's

intro = chosen(INTRO, "the intro avatar video")
intro_audio = chosen(INTRO_AUDIO, "the intro's sound") if INTRO_AUDIO else None
print(f"The intro: {intro.name} ({duration(intro):.1f} s)")
display(Video(str(intro), embed=True, width=270))


## 2. The screen recording

Run the cell, then choose the screen recording on your computer (or set `RECORDING` to a file on Drive).

In [ ]:
RECORDING = None  # None: upload it; or a file on Drive, e.g. HOME / "recordings" / "makable-tight-narrated.mp4"

recording = chosen(RECORDING, "the screen recording")
width, height = frame_size(recording)
print(f"The screen recording: {recording.name} ({duration(recording):.1f} s, {width}x{height})")


## 3. The whole video

The intro, then the screen recording. `SIZE = (1920, 1080)` makes a wide video.

In [ ]:
SIZE = (1080, 1920)  # the video's width and height: (1920, 1080) for a wide one

out = RUN / f"{recording.stem}-with-intro.mp4"
out.parent.mkdir(parents=True, exist_ok=True)
merge(intro, recording, out, intro_audio=intro_audio, size=SIZE)
print(f"Made {out} ({duration(out):.1f} s)")
display(Video(str(out), embed=True, width=270))
